<img src="https://github.com/priyammaz/MyTorch/blob/main/src/banner.png?raw=true" alt="drawing" width="800"/>

## Quick Start Guide!

Lets do a quick demo showcasing what MyTorch can do! First things first, we need to do an installation. For basic installation (with no Torch/Triton dependencies)


### Installation
```bash
git clone https://github.com/priyammaz/MyTorch.git
cd MyTorch && pip install .
```

### Imports

In [1]:
import os
import pickle
import requests
import numpy as np
from dataclasses import dataclass
from tqdm.notebook import tqdm

import mytorch
import mytorch.nn as nn
import mytorch.nn.functional as F
import mytorch.optim as optim
from mytorch.utils.data import Dataset, DataLoader
from mytorch.accelerate import Accelerator


### Hyperparameters

Lets set all the parameters we need for training here!

In [2]:
### MODEL PARAMS ###
MAX_SEQ_LEN = 256
EMBED_DIM = 384
NUM_HEADS = 6
NUM_BLOCKS = 6
MLP_RATIO = 4
DROPOUT_P = 0.0
USE_BIAS = False
USE_FUSED = True

### TRAINING PARAMS ###
TRAINING_ITERATIONS = 2500
EVAL_INTERVAL = 500
EVAL_ITERS = 200
BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 1
MAX_LR_RATE = 1e-3
MIN_LR_RATE = 1e-4
WARMUP_STEPS = 500
WEIGHT_DECAY = 0.1
NUM_WORKERS = 4
DEVICE = "cuda"
LOG_ITER = 250

### ACCELERATOR CONFIG ###
MIXED_PRECISION = True
LOG_WANDB = False

### EXPERIMENT PARAMETERS ###
EXPERIMENT_NAME = "gpt2-tiny"
RUN_NAME = "shakespeare"

### GENERATION CONFIG ###
START = "\n"
GEN_LENGTH = 512
TEMPERATURE = 0.8
TOPK = 30
FP16_GEN = True

### Define a Tiny GPT2

This is an implementation pretty close to [NanoGPT](https://github.com/karpathy/nanoGPT)!

In [3]:
@dataclass
class GPT2Config:

    vocab_size: int =  65
    max_seq_len: int = 256
    embed_dim: int = 384
    mlp_ratio: int = 4
    num_blocks: int = 6
    num_heads: int = 6
    mlp_dropout_p: float = 0.0
    attn_dropout_p: float = 0.0 # Currently not supported for our flash attn
    use_bias: bool = False
    use_full_auto: bool = False # Use only autograd or use manual grad for known composites of ops
    use_fused_ops: bool = False # Only works if triton installed

class Embeddings(nn.Module):

    def __init__(self, vocab_size, embed_dim, context_length, fused):
        super().__init__()
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.context_length = context_length

        ### Embeddings for Tokens ###
        self.embeddings = nn.Embedding(vocab_size, embed_dim, fused=fused)

        ### Positional Embeddings ###
        self.position_embeddings = nn.Embedding(context_length, embed_dim, fused=fused)

    def forward(self, input_ids):

        batch_size, seq_length = input_ids.shape

        ### Convert Tokens to Embeddings ###
        x = self.embeddings(input_ids)

        ### Add Positional Information ###
        avail_idx = mytorch.arange(start=0, end=seq_length).to(input_ids.device)
        pos_embed = self.position_embeddings(avail_idx).reshape(1, seq_length, self.embed_dim)
        x = x + pos_embed

        return x

class Attention(nn.Module):

    def __init__(self,
                 embed_dim,
                 num_heads,
                 attn_dropout_p=0.1,
                 context_length=1024,
                 use_bias=True,
                 auto=False,
                 fused=False):

        super().__init__()

        ### Sanity Checks ###
        assert embed_dim % num_heads == 0, "Double check embedding dim divisible by number of heads"

        ### Attention Head Dim ###
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.fused = fused

        ### Attention Projections ###
        self.qkv_proj = nn.Linear(embed_dim, 3 * embed_dim, bias=use_bias, auto=auto)

        ### Post Attention Projection ###
        self.out_proj = nn.Linear(embed_dim, embed_dim, bias=use_bias, auto=auto)
        self.proj_drop = nn.Dropout(dropout_p=attn_dropout_p)

        if not self.fused:
            self.attn_drop = nn.Dropout(dropout_p=attn_dropout_p) # Currently only support attn dropout in non-fused
            self.softmax = nn.Softmax(auto=auto, fused=fused)

            ### If we are not using fused attention we need to manually pass in our
            ### attention mask! So lets just save it as a buffer right here!
            causal_positions = (mytorch.tril(mytorch.ones((1,1,context_length,context_length))) == 0)
            causal_mask = mytorch.masked_fill(mytorch.zeros((1,1,context_length, context_length)), causal_positions, value=float("-inf"))
            self.register_buffer("causal_mask", causal_mask)

    def forward(self, x):

        batch, seq_len, embed_dim = x.shape

        # QKV projection
        qkv = self.qkv_proj(x)  # [batch, seq_len, 3*embed_dim]

        # Reshape to multi-head
        qkv = qkv.reshape(batch, seq_len, self.num_heads, 3 * self.head_dim)

        # Transpose to [batch, num_heads, seq_len, 3*head_dim]
        qkv = qkv.transpose(1, 2)

        # Chunk last dim into q, k, v
        q, k, v = mytorch.chunk(qkv, 3, dim=-1)  # each [batch, num_heads, seq_len, head_dim]

        if not self.fused:

            # Compute attention scores
            scores = (q @ k.transpose(-2, -1)) / (self.head_dim ** 0.5)

            masked_scores = scores + self.causal_mask[:, :, :seq_len, :seq_len].astype(scores.data.dtype)

            softmax_attention = self.softmax(masked_scores, dim=-1)
            dropped_attention = self.attn_drop(softmax_attention)

            # Attention output
            output = dropped_attention @ v

        else:

            output = F.scaled_dot_product_attention(q, k, v, is_causal=True)

        output = output.transpose(1, 2).reshape(batch, seq_len, embed_dim)

        # Output projection
        output = self.out_proj(output)
        output = self.proj_drop(output)

        return output

class FeedForward(nn.Module):
    """
    Regular MLP module after our attention computation.
    """
    def __init__(self,
                 embed_dim,
                 mlp_ratio=4,
                 mlp_dropout_p=0.1,
                 use_bias=True,
                 auto=False,
                 fused=False):
        super().__init__()

        self.fused = fused
        hidden_size = embed_dim * mlp_ratio

        if fused:
          self.intermediate_dense = nn.Linear(embed_dim, hidden_size, bias=use_bias, auto=auto, fused=fused, act_func="gelu")

        else:
          self.intermediate_dense = nn.Linear(embed_dim, hidden_size, bias=use_bias, auto=auto)
          self.activation = nn.GELU()
        self.intermediate_dropout = nn.Dropout(mlp_dropout_p)

        self.out_proj = nn.Linear(hidden_size, embed_dim, bias=use_bias, auto=auto)
        self.output_dropout = nn.Dropout(mlp_dropout_p)

    def forward(self, x):

        x = self.intermediate_dense(x)
        if not self.fused:
          x = self.activation(x)

        x = self.intermediate_dropout(x)
        x = self.out_proj(x)
        x = self.output_dropout(x)

        return x

class TransformerBlock(nn.Module):
    def __init__(self,
                 embed_dim,
                 num_heads,
                 dropout_p,
                 mlp_ratio=4,
                 context_length=1024,
                 use_bias=True,
                 auto=False,
                 fused=False):

        super().__init__()

        self.attention = Attention(embed_dim=embed_dim,
                                   num_heads=num_heads,
                                   attn_dropout_p=dropout_p,
                                   context_length=context_length,
                                   use_bias=use_bias,
                                   auto=auto,
                                   fused=fused)

        self.layernorm1 = nn.LayerNorm(embed_dim, bias=use_bias, auto=auto, fused=fused)
        self.feedforward = FeedForward(embed_dim, mlp_ratio, dropout_p, use_bias, auto=auto, fused=fused)
        self.layernorm2 = nn.LayerNorm(embed_dim, bias=use_bias, fused=fused)

    def forward(self, x):

        attn_out = self.attention(self.layernorm1(x))
        x = x + attn_out
        mlp_out = self.feedforward(self.layernorm2(x))
        x = x + mlp_out

        return x

class GPT2(nn.Module):
    def __init__(self, config):

        super().__init__()

        self.config = config

        self.embeddings = Embeddings(vocab_size=config.vocab_size,
                                     embed_dim=config.embed_dim,
                                     context_length=config.max_seq_len,
                                     fused=config.use_fused_ops)

        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim=config.embed_dim,
                             num_heads=config.num_heads,
                             dropout_p=config.mlp_dropout_p,
                             mlp_ratio=config.mlp_ratio,
                             context_length=config.max_seq_len,
                             use_bias=config.use_bias,
                             fused=config.use_fused_ops,
                             auto=config.use_full_auto)

            for _ in range(config.num_blocks)
        ])

        self.final_layer_norm = nn.LayerNorm(config.embed_dim,
                                             auto=config.use_full_auto,
                                             fused=config.use_fused_ops)
        self.lm_head = nn.Linear(config.embed_dim,
                                 config.vocab_size,
                                 bias=config.use_bias,
                                 auto=config.use_full_auto,
                                 fused=config.use_fused_ops)

        ### Initialize Weights ###
        self.apply(_init_weights)
        for name, param in self.named_parameters():
            if "out_proj" in name:
                mytorch.nn.init.normal_(param, mean=0.0, std=(0.02/np.sqrt(2 * config.num_blocks)))

        ### Weight tying ###
        self.lm_head.weight = self.embeddings.embeddings.weight

    def forward(self, x):

        x = self.embeddings(x)

        for block in self.blocks:
            x = block(x)

        x = self.final_layer_norm(x)
        x = self.lm_head(x)

        return x

### Standard Weight Init for Transformers ###
def _init_weights(module):
    if isinstance(module, nn.Linear):
        mytorch.nn.init.normal_(module.weight, mean=0, std=0.02)
        if module.bias is not None:
            mytorch.nn.init.zeros_(module.bias)

    elif isinstance(module, nn.Embedding):
        mytorch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    elif isinstance(module, nn.LayerNorm):
        mytorch.nn.init.ones_(module.weight)
        if module.bias is not None:
            mytorch.nn.init.zeros_(module.bias)

### Prepare Data For Training

Most of this code came from again the [NanoGPT](https://github.com/karpathy/nanoGPT/tree/master/data/shakespeare) repo! This will create two ```.bin``` files with all our tokens as well as a ```tokenizer.pkl```

In [4]:
def prep_shakespeare(path_to_store, test_split_pct=0.1):
    data_url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
    all_txt = requests.get(data_url).text
    n = len(all_txt)
    print(f"Loaded {n} Characters of Data")

    ### Get all unique chars ###
    chars = sorted(list(set(all_txt)))
    vocab_size = len(chars)
    print("all the unique characters:", ''.join(chars))
    print(f"vocab size: {vocab_size:,}")

    ### Quick Tokenizers ###
    char2idx = {c:i for i,c in enumerate(chars)}
    idx2char = {i:c for (c,i) in char2idx.items()}

    ### Quick encoding ###
    def encode(s):
        return [char2idx[c] for c in s]

    ### Train/Test Split ###
    train = all_txt[:int(n*0.9)]
    test = all_txt[int(n*0.9):]

    train_ids = encode(train)
    val_ids = encode(test)
    print(f"train has {len(train_ids):,} tokens")
    print(f"val has {len(val_ids):,} tokens")

    ### Store Tokenizer ###
    meta = {
        "vocab_size": vocab_size,
        "char2idx": char2idx,
        "idx2char": idx2char
    }

    ### Store in Bin Files ###
    if not os.path.exists(path_to_store):
        os.makedirs(path_to_store, exist_ok=True)

    path_to_train = os.path.join(path_to_store, "train.bin")
    path_to_test = os.path.join(path_to_store, "val.bin")
    path_to_tokenizer_pkl = os.path.join(path_to_store, "tokenizer.pkl")

    train_ids = np.array(train_ids, dtype=np.uint16)
    val_ids = np.array(val_ids, dtype=np.uint16)
    train_ids.tofile(path_to_train)
    val_ids.tofile(path_to_test)

    with open(path_to_tokenizer_pkl, "wb") as f:
        pickle.dump(meta, f)

os.makedirs("data/", exist_ok=True)
prep_shakespeare("data")

Loaded 1115394 Characters of Data
all the unique characters: 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
vocab size: 65
train has 1,003,854 tokens
val has 111,540 tokens


### Load all the Prepped Files

In [5]:
path_to_train = "data/train.bin"
path_to_test = "data/val.bin"
path_to_tokenizer = "data/tokenizer.pkl"

with open(path_to_tokenizer, "rb") as f:
  tokenizer_meta = pickle.load(f)
vocab_size = tokenizer_meta["vocab_size"]

### Simple DataLoader

In [6]:
class TokenLoader(Dataset):
    def __init__(self, path_to_bin, context_length):

        self.path_to_bin = path_to_bin
        self.context_length = context_length
        self.arr = np.memmap(path_to_bin, dtype=np.uint16, mode='r')
        self.num_tokens = self.arr.shape[0]

    def __len__(self):
        """
        We dont really have the number of "samples" in our data as we just
        grab random sets of consecutive tokens of size context_length. Lets just
        give a good guess!
        """
        return self.num_tokens // self.context_length

    def __getitem__(self, idx):

        start_idx = np.random.randint(low=0, high=self.num_tokens - self.context_length - 1)
        x = self.arr[start_idx:start_idx+self.context_length]
        y = self.arr[start_idx+1:start_idx+self.context_length+1]
        return mytorch.Tensor(x), mytorch.Tensor(y)

trainset = TokenLoader(path_to_train, context_length=MAX_SEQ_LEN)
testset = TokenLoader(path_to_test, context_length=MAX_SEQ_LEN)

micro_batchsize = BATCH_SIZE // GRADIENT_ACCUMULATION_STEPS
trainloader = DataLoader(trainset, batch_size=micro_batchsize, num_workers=NUM_WORKERS)
testloader = DataLoader(testset, batch_size=micro_batchsize, num_workers=NUM_WORKERS)


### Accelerator

The accelerator class does a few things, MultiGPU Training, Mixed Precision Training and Logging! We can't do MultiGPU in colab, but we can leverage the other stuff so lets use it!

If using ```WandB``` you will have to log in!

In [7]:
accelerator = Accelerator(mixed_precision=MIXED_PRECISION,
                          log_wandb=LOG_WANDB,
                          gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS)

if LOG_WANDB:
  accelerator.init_tracker(
      project_name=EXPERIMENT_NAME,
      run_name=RUN_NAME
  )

### Load Model

In [8]:
gpt2config = GPT2Config(
    vocab_size=vocab_size,
    max_seq_len=MAX_SEQ_LEN,
    embed_dim=EMBED_DIM,
    num_heads=NUM_HEADS,
    num_blocks=NUM_BLOCKS,
    attn_dropout_p=DROPOUT_P, # Only works for non-fused attn for now!
    mlp_dropout_p=DROPOUT_P,
    mlp_ratio=MLP_RATIO,
    use_bias=USE_BIAS,
    use_fused_ops=USE_FUSED
)

model = GPT2(gpt2config).to(DEVICE)

total_params = 0
for param in model.parameters():
    if param.requires_grad:
        total_params += np.prod(param.shape)
print("Total Trainable Parameters:", total_params)

Total Trainable Parameters: 10745472


### Load Optimizer and Scheduler

In [9]:
optimizer = optim.AdamW(model.parameters(),
                        lr=MAX_LR_RATE,
                        weight_decay=WEIGHT_DECAY)

scheduler = mytorch.lr_scheduler.CosineLRScheduler(
    optimizer=optimizer, max_lr=MAX_LR_RATE,
    min_lr=MIN_LR_RATE, total_steps=TRAINING_ITERATIONS,
    warmup_steps=WARMUP_STEPS
)

### Prepare Model/Optimizer

The Accelerator class will handle mixed precision training for you! But this means we need to setup some stuff for this to work. The ```accelerator.prepare``` will wrap the model and optimizer to get this to work!

In [10]:
model, optimizer, trainloader, testloader = accelerator.prepare(
    model, optimizer, trainloader, testloader
)

### Define Loss Function

GPT2 uses Cross Entropy so lets do the same!

In [11]:
loss_fn = nn.CrossEntropyLoss()

### Train Model!

In [12]:
pbar = tqdm(range(TRAINING_ITERATIONS))

completed_steps = 0
train = True
while train:

    for inputs, targets in trainloader:

        # Move to correct device
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)

        # Forward pass
        logits = model(inputs)
        loss = loss_fn(logits, targets)

        # Backward
        accelerator.backward(loss)

        # Clip gradients (and get the grads to check on training health)
        grad_norm = accelerator.clip_grad_norm_(1.0)

        # Step optimizer
        optimizer.step()
        optimizer.zero_grad()

        ### Accelerator tracks when accumulation is done, the flag is just sync_grad ###
        if accelerator.sync_grad:

            ### One full accumulation complete ###
            completed_steps += 1
            pbar.update(1)

            ### Update Scheduler ###
            scheduler.step()

            ### Gather metrics across GPUs (Doesnt apply to colab but this is just an example!)
            if completed_steps % LOG_ITER == 0:

                ### Gather (no-op if we are on single GPU) ###
                loss = accelerator.gather_for_metrics(loss)

                ### Grab our stored grad_norm for checking on model health ###
                log_statement = f"Iter {completed_steps}, Loss: {loss:.4f}, LR: {scheduler.get_last_lr()[0]:.2e}"
                if accelerator.grad_norm is not None:
                    log_statement += f" Grad Norm: {accelerator.grad_norm:.3f}"

                print(log_statement)

                ### Log with Wandb if enabled ###
                if LOG_WANDB:
                    logging_dict = {"loss": loss, "lr": scheduler.get_last_lr()[0]}
                    if accelerator.grad_norm is not None:
                        logging_dict["grad_norm"] = accelerator.grad_norm
                    accelerator.log(logging_dict, step=completed_steps)

            if completed_steps % EVAL_INTERVAL == 0:

                accelerator.print("Evaluating!")
                model.eval()

                val_losses = []

                for val_iter, (inputs, targets) in enumerate(testloader):
                    if val_iter >= EVAL_ITERS:
                        break  # stop after desired number of eval iterations

                    inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)

                    with mytorch.no_grad():
                        logits = model(inputs)

                    loss = loss_fn(logits, targets)
                    loss_val = accelerator.gather_for_metrics(loss)
                    val_losses.append(loss_val)

                ### Log Loss ###
                val_losses = np.mean(val_losses)
                print("Validation Loss:", val_losses)
                if LOG_WANDB:
                    logging_dict = {"val_loss": val_losses}
                    accelerator.log(logging_dict, step=completed_steps)

                ### Set back into Training Mode ###
                model.train()

        if completed_steps >= TRAINING_ITERATIONS:
            accelerator.print("Completed Training!!!")
            train = False
            break

### Save Checkpoint! ###
accelerator.save_state(f"final_checkpoint")

  0%|          | 0/2500 [00:00<?, ?it/s]

/home/priyam/anaconda3/envs/mytorch/lib/python3.13/site-packages/triton/language/core.py:2625: UserWarning: tl.make_block_ptr is deprecated. Use TensorDescriptor or tl.make_tensor_descriptor instead.
  warn("tl.make_block_ptr is deprecated. Use TensorDescriptor or tl.make_tensor_descriptor instead.")
/mnt/analysisdrive/analysis/MyTorch/mytorch/accelerate.py:810: UserWarning: Warning: NaN/Inf Grad detected at step 18. Reducing Scale to 32768.0. Skipping step.
  warnings.warn(f"Warning: NaN/Inf Grad detected at step {self.step_counter}.{msg} Skipping step.")


Iter 250, Loss: 2.3906, LR: 5.00e-04 Grad Norm: 0.797
Iter 500, Loss: 1.9121, LR: 1.00e-03 Grad Norm: 0.994
Evaluating!
Validation Loss: 2.041050502232143
Iter 750, Loss: 1.5996, LR: 9.66e-04 Grad Norm: 0.607
Iter 1000, Loss: 1.5547, LR: 8.68e-04 Grad Norm: 0.596
Evaluating!
Validation Loss: 1.6907087053571428
Iter 1250, Loss: 1.3516, LR: 7.22e-04 Grad Norm: 0.553
Iter 1500, Loss: 1.3398, LR: 5.50e-04 Grad Norm: 0.546
Evaluating!
Validation Loss: 1.5575823102678572
Iter 1750, Loss: 1.2695, LR: 3.78e-04 Grad Norm: 0.501
Iter 2000, Loss: 1.2178, LR: 2.32e-04 Grad Norm: 0.589
Evaluating!
Validation Loss: 1.5141252790178572
Iter 2250, Loss: 1.1797, LR: 1.34e-04 Grad Norm: 0.578
Iter 2500, Loss: 1.1172, LR: 1.00e-04 Grad Norm: 0.605
Evaluating!
Validation Loss: 1.4732840401785714
Completed Training!!!
Saving Checkpoint to final_checkpoint


## Load Weights

Loading weights works much the same way in PyTorch. The weights were saved in a safetensors file, so we can load the state_dict and then get our model!

In [13]:
from safetensors.numpy import load_file

model = GPT2(gpt2config)
state_dict = load_file("work_dir/final_checkpoint/model.safetensors")
model.load_state_dict(state_dict)
model = model.to(DEVICE)

if FP16_GEN:
  for name, param in model.named_parameters():
    param.astype(mytorch.float16)

## Inference

We can do some simple inferencing to see if we can create some Shakespeare!

In [14]:
### Set Model to Eval ###
model.eval()

### Load our Tokenizer ###
with open("data/tokenizer.pkl", "rb") as f:
        tokenizer_meta = pickle.load(f)
tokenizer_char2idx = tokenizer_meta["char2idx"]
tokenizer_idx2char = tokenizer_meta["idx2char"]

### Get starting tokens for model ###
generated = [tokenizer_char2idx[c] for c in "h"]

seed = mytorch.Tensor(generated, dtype=mytorch.int32).reshape(1,-1).to(DEVICE)
context = seed

### Start the print off with whatever our input was!
print(START, end="", flush=True)
for _ in range(GEN_LENGTH):
    seed = mytorch.Tensor(
        generated[-MAX_SEQ_LEN:], dtype=mytorch.int32
    ).unsqueeze(0).to(DEVICE)

    with mytorch.no_grad():
        pred = model(seed)

    # Get logits for last position
    last_logits = pred[:, -1, :]

    ### Scale by temperature ###
    last_logits = last_logits / TEMPERATURE

    if TOPK is not None and TOPK > 0:

        # Get top-k indices (the top-k largest logits) ###
        topk_idx = last_logits.argsort(descending=True)[:, :TOPK]

        ### Replace all other logits with -inf ###
        mask = mytorch.full_like(last_logits, float("-inf"))
        mask[:, topk_idx] = last_logits[:, topk_idx]
        last_logits = mask

    # Random sample
    exp_logits = mytorch.exp(last_logits - mytorch.max(last_logits))
    probs = exp_logits / mytorch.sum(exp_logits)
    next_id = mytorch.multinomial(probs, num_samples=1)[0].numpy().item()

    # Append to generated sequence
    generated.append(next_id)

    # Decode and print the latest token
    token = tokenizer_idx2char[next_id]

    print(token, end="", flush=True)


ears and then redress
Shall hear the means for a foot of the beast.
Live itself bows not resigns to be simple
That is my words and that I should so pass.
To true my stand is man! We'll ne'er salute.
Dost that stid thou wouldst k we regal friend;
And therefore her.
She was not this new neither, sures the gods
Of the bloody deep of sweet with her piloce!
The great plants and well substant here fear:
A nearer of our warth beauty with a moon,
More and playing the provost sea for of the bastardy.

Hath Seat:
Sir